In [1]:
from pyspark.sql import SparkSession 

In [3]:
spark=SparkSession.builder.appName('Flights').getOrCreate()
spark

In [4]:
#read data from csv file 

flights=spark.read.csv('flights-larger.csv',sep=',',header=True,inferSchema=True,nullValue='NA')

In [5]:
#number of records 

flights.count()

275000

In [6]:
#first five records 

flights.show()

+---+---+---+-------+------+---+----+------+--------+-----+
|mon|dom|dow|carrier|flight|org|mile|depart|duration|delay|
+---+---+---+-------+------+---+----+------+--------+-----+
| 10| 10|  1|     OO|  5836|ORD| 157|  8.18|      51|   27|
|  1|  4|  1|     OO|  5866|ORD| 466|  15.5|     102| NULL|
| 11| 22|  1|     OO|  6016|ORD| 738|  7.17|     127|  -19|
|  2| 14|  5|     B6|   199|JFK|2248| 21.17|     365|   60|
|  5| 25|  3|     WN|  1675|SJC| 386| 12.92|      85|   22|
|  3| 28|  1|     B6|   377|LGA|1076| 13.33|     182|   70|
|  5| 28|  6|     B6|   904|ORD| 740|  9.58|     130|   47|
|  1| 19|  2|     UA|   820|SFO| 679| 12.75|     123|  135|
|  8|  5|  5|     US|  2175|LGA| 214|  13.0|      71|  -10|
|  5| 27|  5|     AA|  1240|ORD|1197| 14.42|     195|  -11|
|  8| 20|  6|     B6|   119|JFK|1182| 14.67|     198|   20|
|  2|  3|  1|     AA|  1881|JFK|1090| 15.92|     200|   -9|
|  8| 26|  5|     B6|    35|JFK|1028| 20.58|     193|  102|
|  4|  9|  5|     AA|   336|ORD| 733|  2

In [7]:
#column data type 

flights.printSchema

<bound method DataFrame.printSchema of DataFrame[mon: int, dom: int, dow: int, carrier: string, flight: int, org: string, mile: int, depart: double, duration: int, delay: int]>

In [8]:
flights.dtypes

[('mon', 'int'),
 ('dom', 'int'),
 ('dow', 'int'),
 ('carrier', 'string'),
 ('flight', 'int'),
 ('org', 'string'),
 ('mile', 'int'),
 ('depart', 'double'),
 ('duration', 'int'),
 ('delay', 'int')]

In [9]:
flights_drop_column=flights.drop('flight')

In [10]:
#number of record wit missing delay values 

flights_drop_column.filter('delay Is Null').count()


16711

In [11]:
#remove record with missing delay values 
flights_valid_delay=flights_drop_column.filter('delay is not null')

In [15]:
#remove records with missing values in any column and get the vnymber of remaining ones 
flights_none_missing=flights_valid_delay.dropna()
print(flights_none_missing.count())

258289


In [18]:
from pyspark.sql.functions import round 

flights_km=flights_none_missing.withColumn('km',round(flights_none_missing.mile*1.60934,0)).drop('mile')
flights_km=flights_km.withColumn('label',(flights_km.delay>=15).cast('integer'))


flights_km.show(5)

+---+---+---+-------+---+------+--------+-----+------+-----+
|mon|dom|dow|carrier|org|depart|duration|delay|    km|label|
+---+---+---+-------+---+------+--------+-----+------+-----+
| 10| 10|  1|     OO|ORD|  8.18|      51|   27| 253.0|    1|
| 11| 22|  1|     OO|ORD|  7.17|     127|  -19|1188.0|    0|
|  2| 14|  5|     B6|JFK| 21.17|     365|   60|3618.0|    1|
|  5| 25|  3|     WN|SJC| 12.92|      85|   22| 621.0|    1|
|  3| 28|  1|     B6|LGA| 13.33|     182|   70|1732.0|    1|
+---+---+---+-------+---+------+--------+-----+------+-----+
only showing top 5 rows



In [21]:
from pyspark.ml.feature import StringIndexer 

indexer=StringIndexer(inputCol='carrier',outputCol='carrier_idx')

indexer_model=indexer.fit(flights_km)

flights_indexed=indexer_model.transform(flights_km)


flights_indexed=StringIndexer(inputCol='org',outputCol='org_idx').fit(flights_indexed).transform(flights_indexed)
    


In [27]:
from pyspark.ml.feature import VectorAssembler

#CREATE AN ASSEMBLER OBJECT 

assembler=VectorAssembler(inputCols=['mon', 'dom', 'dow',
 'carrier_idx',
 'org_idx',
 'km', 'depart', 'duration'],outputCol='features')

In [30]:
flights_assembled=assembler.transform(flights_indexed)

flights_assembled.select('features','delay').show(5,truncate=False)

+-----------------------------------------+-----+
|features                                 |delay|
+-----------------------------------------+-----+
|[10.0,10.0,1.0,2.0,0.0,253.0,8.18,51.0]  |27   |
|[11.0,22.0,1.0,2.0,0.0,1188.0,7.17,127.0]|-19  |
|[2.0,14.0,5.0,4.0,2.0,3618.0,21.17,365.0]|60   |
|[5.0,25.0,3.0,3.0,5.0,621.0,12.92,85.0]  |22   |
|[3.0,28.0,1.0,4.0,3.0,1732.0,13.33,182.0]|70   |
+-----------------------------------------+-----+
only showing top 5 rows



In [31]:
flights_assembled.count()

258289

In [32]:
flights_train,flights_test=flights_assembled.randomSplit([0.8,0.2],seed=17)

training_ratio=flights_train.count()/ flights_assembled.count()
print(training_ratio)

0.7996856234682855


In [33]:
#Build decision tree 

from pyspark.ml.classification import DecisionTreeClassifier 


tree=DecisionTreeClassifier()
tree_model=tree.fit(flights_train)

In [35]:
prediction=tree_model.transform(flights_test)
prediction.select('label','prediction','probability').show(5,False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|1    |0.0       |[0.6399778472249693,0.36002215277503063]|
|1    |0.0       |[0.6399778472249693,0.36002215277503063]|
|0    |1.0       |[0.32048003147747395,0.679519968522526] |
|1    |1.0       |[0.32048003147747395,0.679519968522526] |
|0    |1.0       |[0.37626262626262624,0.6237373737373737]|
+-----+----------+----------------------------------------+
only showing top 5 rows



In [37]:

prediction.groupBy('label', 'prediction').count().show()
# Calculate the elements of the confusion matrix

TN = prediction.filter('prediction = 0 AND label = prediction').count()
TP = prediction.filter('prediction = 1 AND label = prediction').count()
FN = prediction.filter('prediction = 0 AND label = 1').count()
FP = prediction.filter('prediction = 1 AND label = 0').count()
# Accuracy measures the proportion of correct predictions
accuracy = (TN + TP) / (TN + TP + FN + FP)
print(accuracy)

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0| 9133|
|    0|       0.0|16038|
|    1|       1.0|16982|
|    0|       1.0| 9586|
+-----+----------+-----+

0.638203289588125


In [38]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator


#calculate metrics 

precision=TP/(TP+FP)
recall=TP/(TP+FN)

print('precision={:.2f}\nrecall  ={:.2f}'.format(precision,recall))

precision=0.64
recall  =0.65


In [42]:
#read data from csv file 

dogs=spark.read.csv('work/dog_food (1).csv',sep=',',header=True,inferSchema=True,nullValue='NA')



AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/jovyan/work/work/dog_food (1).csv.